# LongFlow P0 — Stage 0: environment + baseline sanity

**Before running:** Runtime → Change runtime type → **L4 GPU** (T4 works but is slower and lacks bf16).

Goal (from `docs/experiments/p0-steering.md`): install VibeVoice, generate one single-speaker and one 2-speaker sample, **listen to both**, and verify the hook map from `docs/resources.md` §1. Record everything in `experiments/p0_steering/NOTES.md`.

In [ ]:
!nvidia-smi

## 1. Install the pinned fork

Pin: `vibevoice-community/VibeVoice` @ `07cb79fea` (main HEAD, verified 2026-07-05). The install pins `transformers==4.51.3` — if Colab complains about a conflict with its preinstalled version, do Runtime → Restart session, then re-run from the `%cd` cell onward (do NOT re-run pip).

In [ ]:
%cd /content
!git clone https://github.com/vibevoice-community/VibeVoice.git
%cd /content/VibeVoice
!git checkout 07cb79fea
!pip install -q -e .

import transformers
print("transformers:", transformers.__version__)  # expect 4.51.3

## 2. First look at the inference path

Stage 0 is deliberately exploratory — see what the demo script expects and which voice presets ship with the repo before generating.

In [ ]:
!ls demo/
!ls demo/voices/ 2>/dev/null || echo '(no demo/voices dir — check ls output above for where presets live)'
!ls demo/text_examples/ 2>/dev/null || true
!python demo/inference_from_file.py --help

## 3. Baseline generation — single speaker, then 2-speaker dialogue

Weights: [`microsoft/VibeVoice-1.5B`](https://huggingface.co/microsoft/VibeVoice-1.5B) — MIT, ungated, ~5.4 GB, auto-downloads on first use (no HF token needed).

Adjust `--speaker_names` to match the voice preset names printed above (the demo matches names against files in the voices dir).

In [ ]:
single = (
    "Speaker 1: Hello there! This is a quick sanity check of VibeVoice running in Colab.\n"
    "Speaker 1: If this sounds like normal, intelligible speech, the baseline is fine.\n"
)
dialogue = (
    "Speaker 1: Did the model download without any issues?\n"
    "Speaker 2: It did, and both of our voices should sound distinct from each other.\n"
    "Speaker 1: Great — that is everything stage zero needs to show.\n"
)
with open("/content/single.txt", "w") as f:
    f.write(single)
with open("/content/dialogue.txt", "w") as f:
    f.write(dialogue)

# EDIT speaker names per the voices listing above, then run:
!python demo/inference_from_file.py --model_path microsoft/VibeVoice-1.5B \
    --txt_path /content/single.txt --speaker_names Alice
!python demo/inference_from_file.py --model_path microsoft/VibeVoice-1.5B \
    --txt_path /content/dialogue.txt --speaker_names Alice Frank

In [ ]:
# Find whatever the demo wrote and play it. LISTEN — this checkbox is not optional.
import glob
from IPython.display import Audio, display

wavs = sorted(glob.glob("/content/VibeVoice/outputs/**/*.wav", recursive=True)) or sorted(
    glob.glob("/content/**/*.wav", recursive=True)
)
for w in wavs[-4:]:
    print(w)
    display(Audio(w))

## 4. Verify the hook map (deliverable for NOTES.md)

Checks the claims in `docs/resources.md` §1 against the actually-downloaded model: layer module path, hidden size 1536, 28 layers, diffusion-head interface, and the shipped default inference steps (README reconciliation #1).

In [ ]:
import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)

model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    "microsoft/VibeVoice-1.5B", torch_dtype=torch.bfloat16, device_map="cuda"
)

layers = model.model.language_model.layers
cfg = model.config
print("decoder layers:", len(layers), type(layers[0]).__name__)
print("hidden_size:", cfg.decoder_config.hidden_size)  # expect 1536
assert cfg.decoder_config.hidden_size == 1536 and len(layers) == 28

head = model.model.prediction_head
print("head:", type(head).__name__)
print("head params (M):", sum(p.numel() for p in head.parameters()) / 1e6)
print("acoustic latent dim:", cfg.acoustic_vae_dim if hasattr(cfg, "acoustic_vae_dim") else "(check config attrs below)")

# README reconciliation #1: what are the ACTUAL default sampling steps + CFG?
import inspect
sig = inspect.signature(model.sample_speech_tokens) if hasattr(model, "sample_speech_tokens") else None
print("sample_speech_tokens defaults:", sig)
print([a for a in dir(cfg) if "step" in a.lower() or "cfg" in a.lower() or "vae" in a.lower()])

## 5. Record findings

Copy into `experiments/p0_steering/NOTES.md` (Stage 0 checklist):
- weights revision (printed during download), fork commit `07cb79fea`, transformers version
- both samples generated + listened: normal? distinct speakers?
- hook-map asserts passed? head param count? actual default steps/CFG?
- anything surprising in the demo script's inference path

**Save your audio** (Colab disk is wiped): download the wavs or mount Drive and copy them out. Next: Stage 1 — contrast-pair generation (`src/steering/contrast_pairs.py`).